# ReviewSense
## IMDb Movie Review Sentiment Classification

This notebook documents the complete data exploration, text preprocessing,
feature extraction, model comparison, hyperparameter tuning, and evaluation
workflow for ReviewSense.

**Dataset:** IMDb 50K Movie Reviews  
**Task:** Binary sentiment classification (positive / negative)  
**Final model:** Tuned Linear Support Vector Classifier (LinearSVC)

## 1. Problem Statement

Explain the problem of manually analyzing
large collections of customer reviews.

## 2. Project Objectives

- Analyze written reviews.
- Preprocess text data.
- Train sentiment classification models.
- Evaluate model performance.


## 3. Environment Setup

Import the libraries required for the project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 4. Dataset Loading

In [ ]:
import pandas as pd

# Load the original IMDb dataset without silently skipping records.
df = pd.read_csv("IMDB Dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

### Dataset integrity note

The original file contains 50,000 reviews. The standard CSV reader is used so
that malformed rows are not silently skipped. After removing 418 exact
duplicate records, the working dataset contains **49,582 unique reviews**.
If the file cannot be parsed, investigate the source file rather than using
`on_bad_lines="skip"`, which can silently change the experiment dataset.

In [ ]:
print("Dataset shape:", df.shape)

print("\nFirst five records:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

## 5. Data Quality Investigation

In this section, we investigate duplicate records,
empty reviews, and review length distributions.

We also verify whether the dataset contains
unexpectedly skipped or malformed records.

In [ ]:
# Check duplicate rows

print("Duplicate rows:", df.duplicated().sum())

# Check duplicate review texts

print("Duplicate reviews:", df["review"].duplicated().sum())

In [ ]:
# Check empty or whitespace-only reviews

empty_reviews = df["review"].str.strip().eq("").sum()

print("Empty reviews:", empty_reviews)

In [ ]:
# Calculate the number of characters in each review

df["review_length"] = df["review"].str.len()

print(df["review_length"].describe())

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df["review_length"], bins=50)

plt.title("Distribution of Movie Review Lengths")
plt.xlabel("Number of Characters")
plt.ylabel("Number of Reviews")

plt.show()

## 6. Duplicate Record Investigation

We investigate repeated reviews and check whether
duplicate texts have conflicting sentiment labels.

In [ ]:
# Check whether identical reviews have different sentiment labels

label_counts = df.groupby("review")["sentiment"].nunique()

conflicting_reviews = label_counts[label_counts > 1]

print("Reviews with conflicting labels:", len(conflicting_reviews))

In [ ]:
# Keep an untouched copy of the original dataset

df_raw = df.copy()

# Remove exact duplicate records
df = df.drop_duplicates(
    subset=["review", "sentiment"]
).copy()

print("Original dataset:", df_raw.shape)
print("After removing duplicates:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

In [ ]:
label_counts = df.groupby("review")["sentiment"].nunique()

conflicting_reviews = label_counts[label_counts > 1]

print("Reviews with conflicting labels:", len(conflicting_reviews))

# Phase 02: NLP and Text Preprocessing

## 1. Understanding Raw Text

In this phase, we explore the structure of natural
language data and prepare movie reviews for
sentiment classification.

### 1.1 Inspecting Raw Reviews

In [ ]:
# Display three raw movie reviews

for i in range(3):
    print(f"\nReview {i + 1}:")
    print(df["review"].iloc[i])
    print("-" * 80)

In [ ]:
# Count reviews containing HTML break tags

html_count = df["review"].str.contains(
    r"<br\s*/?>",
    case=False,
    regex=True
).sum()

print("Reviews containing HTML break tags:", html_count)

In [ ]:
import re

sample_review = df["review"].iloc[0]

print("Original review:")
print(sample_review)

print("\nContains HTML tags:", bool(re.search(r"<[^>]+>", sample_review)))

print("Contains numbers:", bool(re.search(r"\d", sample_review)))

print("Contains punctuation:", bool(re.search(r"[^\w\s]", sample_review)))

### 1.2 HTML Tag Removal

Movie reviews contain HTML formatting tags such as
<br />, which are not meaningful textual content.

We will investigate and remove these tags while
preserving the original review column.

In [ ]:
# Import regular expression library
import re

# Create a function to remove HTML tags
def remove_html_tags(text):
    clean_text = re.sub(r"<[^>]+>", " ", text)
    return clean_text

# Apply the function to our reviews
df["review_no_html"] = df["review"].apply(remove_html_tags)

# Compare original and cleaned reviews
print("ORIGINAL REVIEW:")
print(df["review"].iloc[0][:500])

print("\nCLEANED REVIEW:")
print(df["review_no_html"].iloc[0][:500])

In [ ]:
# Verify whether HTML tags remain

remaining_html = df["review_no_html"].str.contains(
    r"<[^>]+>",
    regex=True
).sum()

print("Remaining HTML tags:", remaining_html)

### 1.3 Lowercase Normalization

Text may contain uppercase and lowercase versions of the same word.
We convert all text to lowercase to maintain consistency during analysis.

In [ ]:
# Convert cleaned reviews to lowercase
df["review_lower"] = df["review_no_html"].str.lower()

# Compare the results
print("BEFORE LOWERCASE:")
print(df["review_no_html"].iloc[0][:300])

print("\nAFTER LOWERCASE:")
print(df["review_lower"].iloc[0][:300])

### 1.4 Tokenization

Tokenization divides text into smaller units called tokens.
For this project, we begin by splitting reviews into individual words.

In [ ]:
# Select one cleaned and lowercased review
sample_text = df["review_lower"].iloc[0]

# Split the review into individual words
tokens = sample_text.split()

# Display the results
print("Original text:")
print(sample_text[:300])

print("\nFirst 30 tokens:")
print(tokens[:30])

print("\nTotal number of tokens:")
print(len(tokens))

In [ ]:
# Display the first 20 tokens with their positions
for index, word in enumerate(tokens[:20]):
    print(index, ":", word)

In [ ]:
# Import the English stop words list
import nltk
from nltk.corpus import stopwords

# Download English stop words
nltk.download("stopwords")

# Get English stop words
stop_words = set(stopwords.words("english"))

# Display the number of stop words
print("Total English stop words:", len(stop_words))

# Display the first 30 stop words
print("Some stop words:", list(stop_words)[:30])

In [ ]:
# Check important negation words
negation_words = ["not", "no", "nor", "never"]

for word in negation_words:
    print(word, ":", word in stop_words)

In [ ]:
# Create a copy of the original stop-word set
custom_stop_words = stop_words.copy()

# Words that must be preserved for sentiment analysis
negation_words = ["not", "no", "nor", "never"]

# Remove negation words from our stop-word set
custom_stop_words.difference_update(negation_words)

# Verify the changes
for word in negation_words:
    print(word, ":", word in custom_stop_words)

### 1.5 Stop-word Removal

Stop-word removal eliminates selected common words that may contribute
less useful information to our sentiment analysis task.

We use a customized stop-word list to preserve important negation words.

In [ ]:
# Function to remove selected stop words
def remove_stop_words(text):
    words = text.split()

    filtered_words = [
        word for word in words
        if word not in custom_stop_words
    ]

    return " ".join(filtered_words)


# Apply stop-word removal to all reviews
df["review_no_stopwords"] = df["review_lower"].apply(remove_stop_words)


# Compare before and after
print("BEFORE STOP-WORD REMOVAL:")
print(df["review_lower"].iloc[0][:300])

print("\nAFTER STOP-WORD REMOVAL:")
print(df["review_no_stopwords"].iloc[0][:300])

In [ ]:
test_sentence = "i do not like this movie"

result = remove_stop_words(test_sentence)

print("Original:", test_sentence)
print("After removal:", result)

In [ ]:
# Find negation contractions in the original stop-word list
negation_contractions = {
    word for word in stop_words
    if word.endswith("n't")
}

# Remove negations and negation contractions from stop words
custom_stop_words.difference_update(negation_words)
custom_stop_words.difference_update(negation_contractions)

# Check the protected contractions
print("Protected negation contractions:")
print(sorted(negation_contractions))

In [ ]:
test_sentences = [
    "i do not like this movie",
    "i don't like this movie",
    "this movie wasn't good"
]

for sentence in test_sentences:
    print("Original:", sentence)
    print("Processed:", remove_stop_words(sentence))
    print("-" * 50)

### 1.6 Stemming

Stemming reduces words to their root-like forms by removing prefixes
or suffixes using predefined rules.

The resulting stems are not always valid English words.

In [ ]:
# Import the Porter Stemmer
from nltk.stem import PorterStemmer

# Create a stemmer object
stemmer = PorterStemmer()

# Test some words
sample_words = [
    "playing",
    "played",
    "plays",
    "studies",
    "studying",
    "happiness",
    "connected"
]

# Apply stemming
for word in sample_words:
    print(word, "->", stemmer.stem(word))

### 1.7 Lemmatization

Lemmatization reduces words to their valid dictionary base forms,
known as lemmas.

Unlike stemming, it uses linguistic vocabulary information.

In [ ]:
# Import WordNet lemmatizer
from nltk.stem import WordNetLemmatizer

# Download required WordNet resources
nltk.download("wordnet")
nltk.download("omw-1.4")

# Create a lemmatizer object
lemmatizer = WordNetLemmatizer()

# Test sample words
sample_words = [
    "playing",
    "played",
    "studies",
    "studying",
    "happiness",
    "connected",
    "cars"
]

# Apply lemmatization
for word in sample_words:
    print(word, "->", lemmatizer.lemmatize(word))

In [ ]:
# Lemmatize words as verbs
print("VERB LEMMATIZATION:")

for word in ["playing", "played", "studying", "connected"]:
    print(word, "->", lemmatizer.lemmatize(word, pos="v"))


# Lemmatize words as adjectives
print("\nADJECTIVE LEMMATIZATION:")

for word in ["better", "happier"]:
    print(word, "->", lemmatizer.lemmatize(word, pos="a"))

### 1.8 Punctuation Cleaning

Punctuation marks can create unnecessary variations in text.
We remove ordinary punctuation while preserving apostrophes
to protect contractions and negation expressions.

In [ ]:
# Function to remove selected punctuation
def remove_punctuation(text):
    text = re.sub(r"[^\w\s']", "", text)
    return text


# Test the function
sample_text = "This movie is AMAZING!!! I don't like boring movies..."

cleaned_text = remove_punctuation(sample_text)

print("Original:", sample_text)
print("Cleaned:", cleaned_text)

### 1.9 Complete Text Preprocessing Pipeline

We combine our selected text-cleaning operations into one reusable
function to prepare movie reviews for sentiment analysis.

In [ ]:
def preprocess_text(text):

    # 1. Remove HTML tags
    text = remove_html_tags(text)

    # 2. Convert text to lowercase
    text = text.lower()

    # 3. Remove punctuation while preserving apostrophes
    text = remove_punctuation(text)

    # 4. Split text into individual words
    words = text.split()

    # 5. Remove selected stop words
    filtered_words = [
        word for word in words
        if word not in custom_stop_words
    ]

    # 6. Join words back into a string
    cleaned_text = " ".join(filtered_words)

    return cleaned_text

In [ ]:
sample_review = df["review"].iloc[0]

processed_review = preprocess_text(sample_review)

print("ORIGINAL REVIEW:")
print(sample_review[:500])

print("\nPROCESSED REVIEW:")
print(processed_review[:500])

In [ ]:
test_review = "I don't like this movie, it was not good!"

print("Original:", test_review)
print("Processed:", preprocess_text(test_review))

In [ ]:
# Apply our preprocessing function to every review
df["processed_review"] = df["review"].apply(preprocess_text)

# Display the first five processed reviews
df[["review", "processed_review"]].head()

In [ ]:
# Check dataset dimensions
print("Dataset shape:", df.shape)

# Check for missing processed reviews
print("Missing processed reviews:",
      df["processed_review"].isnull().sum())

# Check for empty processed reviews
print("Empty processed reviews:",
      df["processed_review"].str.strip().eq("").sum())

# Check for remaining HTML tags
print("Remaining HTML tags:",
      df["processed_review"].str.contains(
          r"<[^>]+>", regex=True
      ).sum())

In [ ]:
# Calculate review lengths before and after preprocessing
df["original_length"] = df["review"].str.len()
df["processed_length"] = df["processed_review"].str.len()

print("Average original review length:",
      df["original_length"].mean())

print("Average processed review length:",
      df["processed_length"].mean())

In [ ]:
import matplotlib.pyplot as plt

average_lengths = [
    df["original_length"].mean(),
    df["processed_length"].mean()
]

labels = ["Original Reviews", "Processed Reviews"]

plt.figure(figsize=(8, 5))

plt.bar(labels, average_lengths)

plt.title("Average Review Length Before and After Preprocessing")
plt.ylabel("Average Number of Characters")

plt.show()

In [ ]:
df.to_csv("ReviewSense_Preprocessed.csv", index=False)

print("Preprocessed dataset saved successfully!")

In [ ]:
from google.colab import files

files.download("ReviewSense_Preprocessed.csv")


# Phase 3: Feature Extraction (TF-IDF)

## Objective
In this phase, we convert the preprocessed movie reviews into numerical representations that machine learning algorithms can understand.

Since machine learning models cannot directly process raw text, we use TF-IDF (Term Frequency–Inverse Document Frequency) to transform reviews into numerical feature vectors.

## TF-IDF
TF-IDF assigns numerical weights to words based on:
- Term Frequency (TF): How frequently a word appears in a review.
- Inverse Document Frequency (IDF): How informative or rare a word is across the collection of reviews.

## Dataset Preparation
We separate the dataset into:
- Input features (X): Preprocessed movie reviews.
- Target labels (y): Sentiment labels (positive and negative).

The dataset is divided into:
- 70% Training data
- 15% Validation data
- 15% Testing data

Stratified splitting is used to maintain the class distribution across all three datasets.

## Feature Vectorization
We use Scikit-learn's TfidfVectorizer with:
- max_features = 50000
- ngram_range = (1, 2)
- min_df = 2

The vectorizer is fitted only on the training data and then used to transform the validation and testing data to prevent data leakage.

## Sparse Matrix
TF-IDF produces sparse matrices because most word features have zero values for individual reviews. Sparse storage reduces memory usage and improves computational efficiency.


In [ ]:

# Separate input features and target labels

X = df["processed_review"]

y = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())


In [ ]:

from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Temporary set:", X_temp.shape)


In [ ]:

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Testing set:", X_test.shape)


In [ ]:

print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)


In [ ]:

X_train_tfidf = tfidf.fit_transform(X_train)

print("Training TF-IDF shape:", X_train_tfidf.shape)


In [ ]:

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF:", X_train_tfidf.shape)
print("Validation TF-IDF:", X_val_tfidf.shape)
print("Testing TF-IDF:", X_test_tfidf.shape)


In [ ]:

feature_names = tfidf.get_feature_names_out()

print("First 30 features:")
print(feature_names[:30])

print("\nTotal number of features:", len(feature_names))


In [ ]:

print("Matrix type:", type(X_train_tfidf))

print("Total possible values:",
      X_train_tfidf.shape[0] * X_train_tfidf.shape[1])

print("Non-zero values:", X_train_tfidf.nnz)

density = (
    X_train_tfidf.nnz /
    (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
) * 100

print("Matrix density:", round(density, 4), "%")



## Phase 3 Summary

### Results
- Total reviews processed: 49,582
- Training reviews: 34,707
- Validation reviews: 7,437
- Testing reviews: 7,438
- Maximum TF-IDF features: 50,000
- N-gram range: Unigrams and bigrams
- Minimum document frequency: 2

### Sparse Matrix Analysis
- Matrix format: CSR (Compressed Sparse Row)
- Total possible values: 1,735,350,000
- Non-zero values: 4,148,413
- Matrix density: 0.2391%

### Conclusion
The preprocessed movie reviews were successfully converted into numerical TF-IDF feature vectors. The resulting sparse matrices are ready for machine learning model training and evaluation.



# Phase 4: Model Training and Evaluation

## Objective
In this phase, we will train and evaluate machine learning models to classify IMDb movie reviews as positive or negative.

We will use the TF-IDF features generated in Phase 3 as input to our classification models.

## Models to be Compared
1. Logistic Regression
2. Multinomial Naive Bayes
3. Linear Support Vector Machine (SVM)

## Model Evaluation
We will evaluate model performance using:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

## Evaluation Strategy
- Train models using the training dataset.
- Compare and tune models using the validation dataset.
- Select the best-performing model.
- Evaluate the selected model on the untouched test dataset.

**Note:** The test dataset will remain separate throughout model development to prevent data leakage and provide an unbiased final evaluation.



## 4.1 Logistic Regression

Logistic Regression is a supervised machine learning algorithm used for classification tasks.

In ReviewSense, it will learn patterns from TF-IDF features to distinguish between positive and negative movie reviews.

The model will be trained using the training dataset and evaluated using the validation dataset.


In [ ]:

from sklearn.linear_model import LogisticRegression


In [ ]:

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


In [ ]:

logistic_model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed!")


In [ ]:

y_val_pred_lr = logistic_model.predict(X_val_tfidf)

print("First 10 actual labels:")
print(y_val.iloc[:10].values)

print("\nFirst 10 predicted labels:")
print(y_val_pred_lr[:10])


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Calculate accuracy
lr_accuracy = accuracy_score(y_val, y_val_pred_lr)

print("Logistic Regression Accuracy:",
      round(lr_accuracy * 100, 2), "%")

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_lr,
    target_names=["Negative", "Positive"]
))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lr))


### Logistic Regression Results

- Validation accuracy: **89.96%**
- Macro F1-score: approximately **0.90**

These results are from the full-dataset experiment (49,582 unique reviews).

### 4.2 Multinomial Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm commonly used for text classification.

It learns patterns from TF-IDF features and predicts whether a review is positive or negative.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()

In [ ]:
nb_model.fit(X_train_tfidf, y_train)

print("Multinomial Naive Bayes training completed!")

In [ ]:
y_val_pred_nb = nb_model.predict(X_val_tfidf)

print("First 10 actual labels:")
print(y_val.iloc[:10].values)

print("\nFirst 10 predicted labels:")
print(y_val_pred_nb[:10])

In [ ]:
nb_accuracy = accuracy_score(y_val, y_val_pred_nb)

print("Multinomial Naive Bayes Accuracy:",
      round(nb_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_nb,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_nb))

### Multinomial Naive Bayes Results

- Validation accuracy: **88.48%**
- Macro F1-score: approximately **0.88**

These results are from the full-dataset experiment (49,582 unique reviews).

### 4.3 Linear Support Vector Machine (SVM)

Linear SVM is a supervised learning algorithm that finds a decision boundary separating positive and negative reviews.

It works effectively with high-dimensional text features such as TF-IDF.

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
svm_model = LinearSVC(
    random_state=42,
    max_iter=2000
)

In [ ]:
svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM training completed!")

In [ ]:
y_val_pred_svm = svm_model.predict(X_val_tfidf)

print("First 10 actual labels:")
print(y_val.iloc[:10].values)

print("\nFirst 10 predicted labels:")
print(y_val_pred_svm[:10])

In [ ]:
svm_accuracy = accuracy_score(y_val, y_val_pred_svm)

print("Linear SVM Accuracy:",
      round(svm_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_svm,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_svm))

### Linear SVM Results

- Validation accuracy: **90.51%**
- Macro F1-score: approximately **0.91**

Linear SVM achieved the strongest validation performance before hyperparameter tuning.

In [ ]:
import pandas as pd

model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Multinomial Naive Bayes",
        "Linear SVM"
    ],
    "Accuracy": [
        lr_accuracy,
        nb_accuracy,
        svm_accuracy
    ],
    "F1-Score (Macro)": [
        0.90,  # Logistic Regression
        0.88,  # Naive Bayes
        0.91   # Linear SVM
    ]
})

model_results = model_results.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

model_results

In [ ]:
model_results.to_csv(
    "ReviewSense_Model_Comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

In [ ]:
validation_results = pd.DataFrame({
    "Review": X_val,
    "Actual": y_val,
    "Predicted": y_val_pred_svm
})

misclassified = validation_results[
    validation_results["Actual"] != validation_results["Predicted"]
].copy()

print("Total validation errors:", len(misclassified))

misclassified.head(10)

In [ ]:
from sklearn.metrics import f1_score

model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Multinomial Naive Bayes",
        "Linear SVM"
    ],
    "Accuracy": [
        lr_accuracy,
        nb_accuracy,
        svm_accuracy
    ],
    "F1-Score (Macro)": [
        f1_score(y_val, y_val_pred_lr, average="macro"),
        f1_score(y_val, y_val_pred_nb, average="macro"),
        f1_score(y_val, y_val_pred_svm, average="macro")
    ]
})

model_results = model_results.sort_values(
    "Accuracy", ascending=False
).reset_index(drop=True)

model_results.round(4)

In [ ]:
misclassified["Original Review"] = df.loc[
    misclassified.index, "review"
]

misclassified[
    ["Original Review", "Actual", "Predicted"]
].head(5)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 0.5, 1, 2, 5]
}

svm_grid = GridSearchCV(
    LinearSVC(random_state=42, max_iter=3000),
    param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train_tfidf, y_train)

In [ ]:
print("Best Parameters:", svm_grid.best_params_)
print("Best Cross-Validation F1:",
      round(svm_grid.best_score_, 4))

In [ ]:
best_svm = svm_grid.best_estimator_

y_val_pred_tuned = best_svm.predict(X_val_tfidf)

print("Tuned SVM Accuracy:",
      round(accuracy_score(y_val, y_val_pred_tuned) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred_tuned,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_tuned))

### Linear SVM Hyperparameter Tuning

- Best parameter: **C = 0.5**
- Cross-validation macro F1-score: approximately **0.902**
- Tuned validation accuracy: **90.74%**
- Untuned validation accuracy: **90.51%**

The tuned Linear SVM was selected for final evaluation.

In [ ]:
final_model = best_svm

print("Selected Model: Tuned Linear SVM")
print("Best Parameters:", svm_grid.best_params_)

In [ ]:
y_test_pred = final_model.predict(X_test_tfidf)

print("Final Test Accuracy:",
      round(accuracy_score(y_test, y_test_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
import joblib

joblib.dump(final_model, "ReviewSense_SVM_Model.pkl")
joblib.dump(tfidf, "ReviewSense_TFIDF_Vectorizer.pkl")

print("Model and vectorizer saved successfully!")

## Final Model Evaluation

**Selected model:** Tuned Linear SVM (`C = 0.5`)

- Validation accuracy: **90.74%**
- Test accuracy: **91.09%**
- Test macro F1-score: approximately **0.91**

The saved model artifacts were independently checked against the original
50,000-row IMDb dataset after removing 418 duplicate records. Using the
stratified 70/15/15 split (`random_state=42`) and the saved preprocessing
artifacts, the saved LinearSVC reproduced the reported **91.09%** test accuracy
and confusion matrix:

`[[3317, 388], [275, 3458]]`

The test set was held out from model selection. These results describe
performance on the IMDb test split, not guaranteed performance on arbitrary
real-world text.